# Анализ данных супермаркета

## Прикладной проект: Выявление часто покупаемых вместе товаров

В этом ноутбуке мы примем знания вероятности и статистики для анализа реальных данных супермаркета.

### Содержание:
1. Загрузка и предобработка данных
2. Exploratory Data Analysis (EDA)
3. Анализ корреляций
4. Вероятностный анализ
5. Выявление часто покупаемых вместе товаров
6. Рекомендации на основе вероятностей

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Библиотеки успешно загружены!')

## 1. Загрузка и предобработка данных

Для этого проекта мы сгенерируем синтетические данные, имитирующие реальные транзакции супермаркета.

### Структура данных:
- **transaction_id**: Уникальный идентификатор транзакции
- **customer_id**: Идентификатор клиента
- **product**: Название продукта
- **category**: Категория продукта
- **price**: Цена продукта
- **quantity**: Количество
- **timestamp**: Время покупки

In [ ]:
# Генерация синтетических данных супермаркета
np.random.seed(42)

# Продукты и категории
products = {
    'Молочные продукты': ['Молоко', 'Сыр', 'Йогурт', 'Масло', 'Сметана'],
    'Хлебобулочные': ['Хлеб', 'Булочка', 'Батон', 'Круассан', 'Пирожок'],
    'Мясо': ['Курица', 'Говядина', 'Свинина', 'Индейка', 'Колбаса'],
    'Овощи': ['Помидоры', 'Огурцы', 'Картофель', 'Морковь', 'Лук'],
    'Фрукты': ['Яблоки', 'Бананы', 'Апельсины', 'Виноград', 'Груши'],
    'Напитки': ['Сок', 'Вода', 'Лимонад', 'Чай', 'Кофе'],
    'Сладости': ['Шоколад', 'Печенье', 'Торт', 'Конфеты', 'Мороженое']
}

# Вероятности покупки категорий
category_probs = [0.20, 0.15, 0.15, 0.15, 0.15, 0.10, 0.10]

# Генерация транзакций
n_transactions = 1000
transactions = []

for trans_id in range(1, n_transactions + 1):
    customer_id = np.random.randint(1, 201)  # 200 уникальных клиентов
    n_items = np.random.choice([1, 2, 3, 4, 5], p=[0.2, 0.3, 0.25, 0.15, 0.1])
    
    # Выбор категорий
    categories = np.random.choice(list(products.keys()), size=n_items, 
                                  p=category_probs, replace=False)
    
    for category in categories:
        product = np.random.choice(products[category])
        price = np.random.uniform(50, 500) if category != 'Напитки' else np.random.uniform(30, 200)
        quantity = np.random.choice([1, 2, 3], p=[0.6, 0.3, 0.1])
        
        transactions.append({
            'transaction_id': trans_id,
            'customer_id': customer_id,
            'product': product,
            'category': category,
            'price': round(price, 2),
            'quantity': quantity
        })

# Создание DataFrame
df = pd.DataFrame(transactions)

print('Данные супермаркета')
print('=' * 60)
print(f'Количество транзакций: {df["transaction_id"].nunique()}')
print(f'Количество клиентов: {df["customer_id"].nunique()}')
print(f'Количество продуктов: {df["product"].nunique()}')
print(f'Количество записей: {len(df)}')
print(f'\nПервые 10 строк:')
print(df.head(10))

## 2. Exploratory Data Analysis (EDA)

Проведём разведочный анализ данных.

In [ ]:
# EDA
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. Распределение по категориям
category_counts = df['category'].value_counts()
axes[0, 0].barh(category_counts.index, category_counts.values, color='steelblue', alpha=0.7)
axes[0, 0].set_xlabel('Количество покупок')
axes[0, 0].set_title('Распределение по категориям')

# 2. Топ-10 самых популярных продуктов
product_counts = df['product'].value_counts().head(10)
axes[0, 1].barh(product_counts.index, product_counts.values, color='coral', alpha=0.7)
axes[0, 1].set_xlabel('Количество покупок')
axes[0, 1].set_title('Топ-10 продуктов')

# 3. Распределение цен
axes[1, 0].hist(df['price'], bins=30, color='green', alpha=0.7, edgecolor='black')
axes[1, 0].set_xlabel('Цена (руб.)')
axes[1, 0].set_ylabel('Количество')
axes[1, 0].set_title('Распределение цен')

# 4. Количество товаров в транзакции
items_per_transaction = df.groupby('transaction_id').size()
axes[1, 1].hist(items_per_transaction, bins=range(1, 7), color='purple', 
                alpha=0.7, edgecolor='black', align='left')
axes[1, 1].set_xlabel('Количество товаров')
axes[1, 1].set_ylabel('Количество транзакций')
axes[1, 1].set_title('Количество товаров в транзакции')

plt.tight_layout()
plt.show()

# Статистики
print('\nСтатистики:')
print('=' * 60)
print(f'Средняя цена: {df["price"].mean():.2f} руб.')
print(f'Медианная цена: {df["price"].median():.2f} руб.')
print(f'Среднее количество товаров в транзакции: {items_per_transaction.mean():.2f}')
print(f'\nОбщая выручка: {(df["price"] * df["quantity"]).sum():,.2f} руб.')

## 3. Анализ корреляций

Исследуем, какие категории товаров покупаются вместе.

In [ ]:
# Анализ корреляций между категориями

# Создание матрицы совместных покупок
categories = list(products.keys())
n_categories = len(categories)
co_occurrence = pd.DataFrame(0, index=categories, columns=categories)

# Подсчёт совместных покупок
for trans_id, group in df.groupby('transaction_id'):
    trans_categories = group['category'].unique()
    
    for i, cat1 in enumerate(trans_categories):
        for j, cat2 in enumerate(trans_categories):
            if i <= j:
                co_occurrence.loc[cat1, cat2] += 1
                if cat1 != cat2:
                    co_occurrence.loc[cat2, cat1] += 1

# Нормализация
total_transactions = df['transaction_id'].nunique()
co_occurrence_norm = co_occurrence / total_transactions

print('Матрица совместных покупок (вероятности):')
print(co_occurrence_norm.round(3))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Тепловая карта
im = axes[0].imshow(co_occurrence_norm.values, cmap='YlOrRd', vmin=0, vmax=1)
axes[0].set_xticks(range(n_categories))
axes[0].set_xticklabels(categories, rotation=45, ha='right')
axes[0].set_yticks(range(n_categories))
axes[0].set_yticklabels(categories)
axes[0].set_title('Матрица совместных покупок')
plt.colorbar(im, ax=axes[0])

# Добавим значения
for i in range(n_categories):
    for j in range(n_categories):
        axes[0].text(j, i, f'{co_occurrence_norm.values[i, j]:.2f}',
                    ha='center', va='center', fontsize=8)

# Корреляционная матрица
# Создание бинарной матрицы покупок
binary_matrix = pd.DataFrame(0, index=df['transaction_id'].unique(), columns=categories)
for trans_id, group in df.groupby('transaction_id'):
    for cat in group['category'].unique():
        binary_matrix.loc[trans_id, cat] = 1

corr_matrix = binary_matrix.corr()

im = axes[1].imshow(corr_matrix.values, cmap='coolwarm', vmin=-1, vmax=1)
axes[1].set_xticks(range(n_categories))
axes[1].set_xticklabels(categories, rotation=45, ha='right')
axes[1].set_yticks(range(n_categories))
axes[1].set_yticklabels(categories)
axes[1].set_title('Корреляционная матрица')
plt.colorbar(im, ax=axes[1])

plt.tight_layout()
plt.show()

## 4. Вероятностный анализ

Вычислим условные вероятности и поддержку для пар продуктов.

In [ ]:
# Вероятностный анализ

# Подсчёт частоты пар продуктов
pair_counts = Counter()
product_counts = Counter()

for trans_id, group in df.groupby('transaction_id'):
    products_in_trans = group['product'].unique()
    
    # Обновляем счётчики продуктов
    for product in products_in_trans:
        product_counts[product] += 1
    
    # Обновляем счётчики пар
    for pair in combinations(sorted(products_in_trans), 2):
        pair_counts[pair] += 1

# Вычисление метрик
total_trans = df['transaction_id'].nunique()

# Топ-20 пар
top_pairs = pair_counts.most_common(20)

print('Вероятностный анализ пар продуктов')
print('=' * 80)
print(f'{"Пара":<40} {"Частота":>8} {"P(A∩B)":>10} {"P(A|B)":>10} {"P(B|A)":>10}')
print('-' * 80)

for (prod1, prod2), count in top_pairs:
    p_both = count / total_trans
    p_a = product_counts[prod1] / total_trans
    p_b = product_counts[prod2] / total_trans
    p_a_given_b = p_both / p_b
    p_b_given_a = p_both / p_a
    
    pair_name = f'{prod1} + {prod2}'
    print(f'{pair_name:<40} {count:>8} {p_both:>10.4f} {p_a_given_b:>10.4f} {p_b_given_a:>10.4f}')

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Топ-10 пар
top_10_pairs = top_pairs[:10]
pair_names = [f'{p1}\n+ {p2}' for (p1, p2), _ in top_10_pairs]
pair_freqs = [count for _, count in top_10_pairs]

y_pos = np.arange(len(pair_names))
axes[0].barh(y_pos, pair_freqs, color='steelblue', alpha=0.7)
axes[0].set_yticks(y_pos)
axes[0].set_yticklabels(pair_names, fontsize=8)
axes[0].set_xlabel('Частота')
axes[0].set_title('Топ-10 пар продуктов')
axes[0].invert_yaxis()

# Условные вероятности
cond_probs = []
for (prod1, prod2), count in top_10_pairs:
    p_both = count / total_trans
    p_b = product_counts[prod2] / total_trans
    cond_probs.append(p_both / p_b)

axes[1].barh(y_pos, cond_probs, color='coral', alpha=0.7)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(pair_names, fontsize=8)
axes[1].set_xlabel('P(B|A)')
axes[1].set_title('Условные вероятности')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 5. Правила ассоциации (Association Rules)

### Метрики:
- **Поддержка (Support):** $P(A \cap B)$ — как часто A и B встречаются вместе
- **Достоверность (Confidence):** $P(B|A)$ — вероятность B при покупке A
- **Лифт (Lift):** $\frac{P(B|A)}{P(B)}$ — насколько увеличивается вероятность B при покупке A

### Интерпретация:
- **Lift > 1**: Положительная ассоциация (товары покупаются вместе чаще)
- **Lift = 1**: Нет связи
- **Lift < 1**: Отрицательная ассоциация (товары покупаются вместе реже)

In [ ]:
# Правила ассоциации

# Вычисление lift для всех пар
association_rules = []

for (prod1, prod2), count in pair_counts.items():
    p_both = count / total_trans
    p_a = product_counts[prod1] / total_trans
    p_b = product_counts[prod2] / total_trans
    
    confidence = p_both / p_a
    lift = confidence / p_b
    
    association_rules.append({
        'antecedent': prod1,
        'consequent': prod2,
        'support': p_both,
        'confidence': confidence,
        'lift': lift
    })

rules_df = pd.DataFrame(association_rules)

# Фильтрация по поддержке и достоверности
min_support = 0.01
min_confidence = 0.1

filtered_rules = rules_df[
    (rules_df['support'] >= min_support) & 
    (rules_df['confidence'] >= min_confidence)
].sort_values('lift', ascending=False)

print('Правила ассоциации')
print('=' * 80)
print(f'Минимальная поддержка: {min_support}')
print(f'Минимальная достоверность: {min_confidence}')
print(f'\nНайдено правил: {len(filtered_rules)}')
print(f'\nТоп-10 правил по lift:')
print(filtered_rules.head(10).to_string(index=False))

# Визуализация
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot: Support vs Confidence
scatter = axes[0].scatter(filtered_rules['support'], filtered_rules['confidence'], 
                         c=filtered_rules['lift'], cmap='YlOrRd', alpha=0.7, s=50)
axes[0].set_xlabel('Support (P(A∩B))')
axes[0].set_ylabel('Confidence (P(B|A))')
axes[0].set_title('Правила ассоциации')
plt.colorbar(scatter, ax=axes[0], label='Lift')

# Топ-10 по lift
top_10_rules = filtered_rules.head(10)
rule_names = [f'{row["antecedent"]} → {row["consequent"]}' for _, row in top_10_rules.iterrows()]
lift_values = top_10_rules['lift'].values

y_pos = np.arange(len(rule_names))
axes[1].barh(y_pos, lift_values, color='steelblue', alpha=0.7)
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(rule_names, fontsize=8)
axes[1].set_xlabel('Lift')
axes[1].set_title('Топ-10 правил по Lift')
axes[1].axvline(1, color='red', linestyle='--', linewidth=2, label='Lift = 1')
axes[1].legend()
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

## 6. Рекомендательная система

На основе выявленных ассоциаций создадим простую рекомендательную систему.

In [ ]:
# Простая рекомендательная система

def recommend_products(product, rules_df, top_n=5):
    """Рекомендация продуктов на основе правил ассоциации"""
    # Правила, где product является антецедентом
    recommendations = rules_df[rules_df['antecedent'] == product].sort_values('lift', ascending=False)
    
    if len(recommendations) == 0:
        # Попробуем найти правила, где product является консеквентом
        recommendations = rules_df[rules_df['consequent'] == product].sort_values('lift', ascending=False)
        recommendations = recommendations.rename(columns={'antecedent': 'consequent', 'consequent': 'antecedent'})
    
    return recommendations.head(top_n)[['consequent', 'support', 'confidence', 'lift']]

# Пример рекомендаций
test_products = ['Молоко', 'Хлеб', 'Курица', 'Яблоки']

print('Рекомендательная система')
print('=' * 80)

for product in test_products:
    print(f'\nЕсли купили "{product}", рекомендуем:')
    recommendations = recommend_products(product, filtered_rules)
    if len(recommendations) > 0:
        for _, row in recommendations.iterrows():
            print(f'  - {row["consequent"]} (lift: {row["lift"]:.2f}, confidence: {row["confidence"]:.2f})')
    else:
        print('  Нет рекомендаций')

# Визуализация рекомендаций для Молока
fig, ax = plt.subplots(figsize=(10, 6))

product = 'Молоко'
recommendations = recommend_products(product, filtered_rules, top_n=10)

if len(recommendations) > 0:
    y_pos = np.arange(len(recommendations))
    ax.barh(y_pos, recommendations['lift'].values, color='steelblue', alpha=0.7)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(recommendations['consequent'].values)
    ax.set_xlabel('Lift')
    ax.set_title(f'Рекомендации для "{product}"')
    ax.axvline(1, color='red', linestyle='--', linewidth=2, label='Lift = 1')
    ax.legend()
    ax.invert_yaxis()

plt.tight_layout()
plt.show()

## Упражнения

### Упражнение 1: Анализ данных
1. Вычислите средний чек для каждого клиента
2. Найдите топ-5 самых прибыльных клиентов
3. Постройте распределение среднего чека

### Упражнение 2: Временной анализ
1. Проанализируйте, какие категории товаров покупаются чаще в разное время дня
2. Есть ли сезонность в покупках?

### Упражнение 3: Сегментация клиентов
1. Сегментируйте клиентов по частоте покупок
2. Проанализируйте предпочтения разных сегментов

### Упражнение 4: Расширенные правила ассоциации
1. Найдите правила для троек продуктов (A ∩ B → C)
2. Какова минимальная поддержка для троек?
3. Сравните с парами

---

**Решения** можно найти в ноутбуке `solutions/15_Solutions.ipynb`